# C10-competition-craft — Practice p09 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
ks = np.array([3, 5, 7, 9, 11])

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()

X_work, X_sim, y_work, y_sim = train_test_split(
    X, y, test_size=200, random_state=SEED, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_work, y_work, test_size=100, random_state=SEED, stratify=y_work
)

scores = []
for k in ks:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ]).fit(X_tr, y_tr)
    scores.append(f1_score(y_val, candidate.predict(X_val), average="macro"))

val_f1s = np.array(scores, dtype=float)
best_k = int(ks[np.argmax(val_f1s)])
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
]).fit(X_work, y_work)
rehearsal_f1 = float(f1_score(y_sim, final_pipe.predict(X_sim), average="macro"))
optimism_gap = float(val_f1s.max() - rehearsal_f1)
(val_f1s, best_k, rehearsal_f1, optimism_gap)

The positive `optimism_gap` is expected because taking the maximum of noisy validation estimates turns selection noise into apparent skill. This rehearsal is protocol-clean because `X_sim` is an ordinary training-table carve held untouched through selection and evaluated exactly once; it is not the real grading split.

### Answer check

In [ ]:
expected = np.array([0.7612847222222222, 0.8382051558623664,
                     0.8480902777777778, 0.8362266622993777,
                     0.8580631073261273])
assert X_work.shape == (400, 12) and X_sim.shape == (200, 12)
assert X_tr.shape == (300, 12) and X_val.shape == (100, 12)
assert val_f1s.shape == (5,)
assert np.allclose(val_f1s, expected, atol=1e-12, rtol=0)
assert best_k == 11
assert np.isclose(rehearsal_f1, 0.8034719947592532, atol=1e-12, rtol=0)
assert np.isclose(optimism_gap, 0.0545911125668741, atol=1e-12, rtol=0)
assert optimism_gap > 0